# Project — End-to-End RNN Sentence Sentiment Classification

**Goal:** classify one raw review sentence as positive or negative using TensorFlow recurrent neural networks.

This project deliberately keeps preprocessing inside the model so the saved artifact accepts a **raw string**, not a pre-tokenized tensor.

```text
data → validation → EDA → split → baseline
→ training-only vocabulary → SimpleRNN/LSTM/GRU
→ validation model selection → untouched test
→ domain slices → errors → save/reload → monitoring
```

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score
)

SEED=42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT=Path("RNN/projects/sentence_sentiment_classification")
DATA=Path("RNN/data/sentence_sentiment_uci.csv")
(PROJECT/"artifacts").mkdir(parents=True,exist_ok=True)
(PROJECT/"reports").mkdir(parents=True,exist_ok=True)

print("TensorFlow:",tf.__version__)
print("device(s):",[d.device_type for d in tf.config.list_logical_devices()])

## 1. Load and validate the data contract

We validate labels, missing values, source domains and duplicates before modeling. A model cannot compensate for an undefined data contract.

In [ ]:
df=pd.read_csv(DATA)
assert set(df.columns)=={"source","text","label"}
assert set(df["label"].unique())=={0,1}
assert set(df["source"].unique())=={"amazon","imdb","yelp"}
assert not df[["source","text","label"]].isna().any().any()

print("rows:",len(df))
print("duplicate sentences:",int(df["text"].duplicated().sum()))
print(df.groupby(["source","label"]).size().unstack())
display(df.sample(8,random_state=SEED))

## 2. Exploratory analysis: sentence length

RNN compute grows with sequence length. We inspect word counts before selecting a truncation/padding length rather than choosing it blindly.

In [ ]:
df["words"]=df["text"].astype(str).str.split().str.len()
print(df["words"].describe(percentiles=[.5,.75,.9,.95,.99]).round(2))
print(df.groupby("source")["words"].agg(["mean","median","max"]).round(2))

plt.figure(figsize=(8,3))
plt.hist(df["words"],bins=35)
plt.axvline(df["words"].quantile(.95),ls="--",label="95th percentile")
plt.xlabel("words per sentence"); plt.ylabel("sentences"); plt.title("Sentence-length distribution")
plt.legend(); plt.show()

## 3. Train / validation / test contract

The split is stratified by **source + label**, preserving both sentiment balance and domain representation.

The test set remains untouched until the recurrent model has been selected.

In [ ]:
df["stratum"]=df["source"]+"_"+df["label"].astype(str)
train,temp=train_test_split(df,test_size=.30,random_state=SEED,stratify=df["stratum"])
val,test=train_test_split(temp,test_size=.50,random_state=SEED,stratify=temp["stratum"])

for name,part in [("train",train),("validation",val),("test",test)]:
    print("\n",name,len(part))
    print(part.groupby(["source","label"]).size().unstack())

## 4. Non-recurrent baseline

A recurrent model must earn its complexity. We use TF-IDF + logistic regression as a strong, transparent bag-of-words baseline.

This baseline ignores token order beyond its local n-grams, while an RNN explicitly updates state over the sequence.

In [ ]:
tfidf=TfidfVectorizer(max_features=6000,ngram_range=(1,2),min_df=2)
Xtr=tfidf.fit_transform(train["text"])
Xv=tfidf.transform(val["text"])
baseline=LogisticRegression(max_iter=1000,random_state=SEED)
baseline.fit(Xtr,train["label"])
val_base=baseline.predict(Xv)
baseline_val_acc=accuracy_score(val["label"],val_base)
print("TF-IDF logistic validation accuracy:",round(baseline_val_acc,4))

## 5. Training-only vocabulary

The vocabulary is adapted only on training sentences. Unknown validation/test words become `[UNK]`; they never influence feature construction.

In [ ]:
MAX_TOKENS=6000
SEQ_LEN=50

adapt_layer=tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
    standardize="lower_and_strip_punctuation",
)
adapt_layer.adapt(train["text"].astype(str).to_numpy())
vocabulary=adapt_layer.get_vocabulary()

lengths=train["words"].to_numpy()
print("vocabulary size:",len(vocabulary))
print("sequence length:",SEQ_LEN)
print("training sentences longer than sequence limit:",int((lengths>SEQ_LEN).sum()))

## 6. Compare SimpleRNN, LSTM and GRU

Only the recurrent cell changes. Tokenization, embedding size, hidden width, data split and optimization contract remain fixed.

That turns the comparison into an interpretable experiment.

In [ ]:
def build_model(kind):
    Layer={"SimpleRNN":tf.keras.layers.SimpleRNN,
           "LSTM":tf.keras.layers.LSTM,
           "GRU":tf.keras.layers.GRU}[kind]

    vectorizer=tf.keras.layers.TextVectorization(
        max_tokens=len(vocabulary),
        output_mode="int",
        output_sequence_length=SEQ_LEN,
        standardize="lower_and_strip_punctuation",
        name="text_vectorization",
    )
    vectorizer.set_vocabulary(vocabulary[2:])

    inputs=tf.keras.Input(shape=(),dtype=tf.string,name="sentence")
    x=vectorizer(inputs)
    x=tf.keras.layers.Embedding(len(vocabulary),32,mask_zero=True,name="embedding")(x)
    x=Layer(32,name=kind.lower())(x)
    x=tf.keras.layers.Dropout(.20)(x)
    outputs=tf.keras.layers.Dense(1,activation="sigmoid",name="positive_probability")(x)

    model=tf.keras.Model(inputs,outputs,name=f"{kind.lower()}_sentence_classifier")
    model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
    return model

models={}
validation_scores={}
histories={}

for kind in ["SimpleRNN","LSTM","GRU"]:
    tf.keras.utils.set_random_seed(SEED)
    model=build_model(kind)
    history=model.fit(
        train["text"].astype(str).to_numpy(),
        train["label"].to_numpy(),
        validation_data=(val["text"].astype(str).to_numpy(),val["label"].to_numpy()),
        epochs=10,batch_size=64,verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",patience=2,restore_best_weights=True
        )]
    )
    prob=model.predict(val["text"].astype(str).to_numpy(),verbose=0).ravel()
    pred=(prob>=.5).astype(int)
    validation_scores[kind]=accuracy_score(val["label"],pred)
    histories[kind]=history.history
    models[kind]=model
    print(kind,"validation accuracy=",round(validation_scores[kind],4),
          "epochs=",len(history.history["loss"]))

selected=max(validation_scores,key=validation_scores.get)
model=models[selected]
print("\nSelected using validation only:",selected)

In [ ]:
plt.figure(figsize=(8,4))
for kind,h in histories.items():
    plt.plot(h["val_accuracy"],marker="o",label=kind)
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.title("Recurrent model comparison")
plt.legend(); plt.show()

## 7. Untouched final test evaluation

Only now do we use the held-out test set.

In [ ]:
test_prob=model.predict(test["text"].astype(str).to_numpy(),verbose=0).ravel()
test_pred=(test_prob>=.5).astype(int)
acc=accuracy_score(test["label"],test_pred)
precision,recall,f1,_=precision_recall_fscore_support(
    test["label"],test_pred,average="binary",zero_division=0
)
auc=roc_auc_score(test["label"],test_prob)

print(f"selected model: {selected}")
print(f"test accuracy={acc:.4f} precision={precision:.4f} recall={recall:.4f} f1={f1:.4f} roc_auc={auc:.4f}")
print(classification_report(test["label"],test_pred,target_names=["negative","positive"],digits=3))

cm=confusion_matrix(test["label"],test_pred)
plt.figure(figsize=(4,4)); plt.imshow(cm)
plt.xticks([0,1],["neg","pos"]); plt.yticks([0,1],["neg","pos"])
plt.xlabel("predicted"); plt.ylabel("actual"); plt.title("Untouched test confusion matrix")
for i in range(2):
    for j in range(2): plt.text(j,i,cm[i,j],ha="center",va="center")
plt.show()

## 8. Domain slices and error analysis

Aggregate accuracy can hide domain-specific weakness. Because the dataset has Amazon, IMDb and Yelp, we inspect each source separately.

In [ ]:
result=test[["source","text","label","words"]].copy()
result["prob_positive"]=test_prob
result["prediction"]=test_pred
result["correct"]=result["label"]==result["prediction"]
result["confidence"]=np.where(result["prediction"]==1,result["prob_positive"],1-result["prob_positive"])

domain_metrics=[]
for source,g in result.groupby("source"):
    domain_metrics.append({
        "source":source,
        "rows":len(g),
        "accuracy":accuracy_score(g["label"],g["prediction"]),
    })
domain_df=pd.DataFrame(domain_metrics)
display(domain_df.round(4))

errors=result[~result["correct"]].sort_values("confidence",ascending=False)
print("test errors:",len(errors))
display(errors[["source","text","label","prediction","prob_positive","confidence"]].head(12))

## 9. Serialize the full raw-string model

Because `TextVectorization` is inside the Keras graph, the saved artifact receives the same raw sentence contract used during training. We do not need a separate tokenizer file at inference time.

In [ ]:
artifact=PROJECT/"artifacts/best_sentence_rnn.keras"
model.save(artifact)

metrics={
    "selected_model":selected,
    "tfidf_logistic_validation_accuracy":float(baseline_val_acc),
    "validation_accuracy":float(validation_scores[selected]),
    "test_accuracy":float(acc),
    "test_precision":float(precision),
    "test_recall":float(recall),
    "test_f1":float(f1),
    "test_roc_auc":float(auc),
    "vocabulary_size":len(vocabulary),
    "sequence_length":SEQ_LEN,
}
(PROJECT/"reports/metrics.json").write_text(json.dumps(metrics,indent=2))
result.to_csv(PROJECT/"reports/test_predictions.csv",index=False)
domain_df.to_csv(PROJECT/"reports/domain_metrics.csv",index=False)

reloaded=tf.keras.models.load_model(artifact)
check_sentences=np.array([
    "I absolutely loved this and would buy it again.",
    "This was a terrible waste of money."
],dtype=str)
check=reloaded.predict(check_sentences,verbose=0).ravel()
for sentence,p in zip(check_sentences,check):
    print(f"{p:.3f} positive | {sentence}")
print("saved:",artifact)

## 10. What should be monitored in production?

A text classifier can drift even when its API still works.

Monitor at least:

- **sentence-length distribution** — input format or user behavior may change;
- **unknown-token rate** — new vocabulary, products, slang or domains may appear;
- **predicted class mix** — sudden output imbalance can signal upstream change;
- **confidence distribution** — the model may become systematically less certain;
- **quality by source/domain** when delayed labels become available;
- **high-confidence errors** — these are especially useful for targeted retraining.

## Final engineering lesson

RNNs are not only forecasting models. In NLP, recurrence means:

`token at t + previous hidden state → updated linguistic representation`.

For sentence classification, the final recurrent representation summarizes the ordered token sequence and feeds a decision layer.